# Demo: Cylinder Skeletonization


In [8]:
from mascaf import MeshManager, SkeletonGraph, fit_morphology, FitOptions

import logging

logging.basicConfig(level=logging.DEBUG)

## Create and Visualize Cylinder

In [9]:
# Load cylinder mesh

mm = MeshManager(mesh_path="../../data/demo/cylinder.obj")
mm.print_mesh_analysis()
skel = SkeletonGraph.from_txt(f"../../data/demo/cylinder.polylines.txt")
mm.visualize_mesh_3d(title="Original Cylinder", skel=skel)

INFO:mascaf.mesh:Loaded mesh: 34 vertices, 64 faces
INFO:mascaf.mesh:Mesh Analysis Report
INFO:mascaf.mesh:====================
INFO:mascaf.mesh:
Geometry:
INFO:mascaf.mesh:  * Vertices: 34
INFO:mascaf.mesh:  * Faces: 64
INFO:mascaf.mesh:  * Components: 1
INFO:mascaf.mesh:  * Volume: 30.61
INFO:mascaf.mesh:  * Bounds: [-1.0, -1.0, -5.0] to [1.0, 1.0, 5.0]
INFO:mascaf.mesh:
Mesh Quality:
INFO:mascaf.mesh:  * Watertight: True
INFO:mascaf.mesh:  * Winding Consistent: True
INFO:mascaf.mesh:  * Normal Direction: outward
INFO:mascaf.mesh:  * Duplicate Vertices: 0
INFO:mascaf.mesh:  * Degenerate Faces: 0
INFO:mascaf.mesh:
Topology:
INFO:mascaf.mesh:  * Genus: 0
INFO:mascaf.mesh:  * Euler Characteristic: 2
INFO:mascaf.mesh:
No issues detected
INFO:mascaf.mesh:
Recommendation:
INFO:mascaf.mesh:  Mesh appears to be in good condition.
INFO:mascaf.mesh:====================


## Fit Morphology model to Mesh

In [10]:
morph = fit_morphology(mm, skel, options=FitOptions(spacing=1))
morph.to_swc_file("../../data/demo/cylinder.swc")
morph.print_attributes()

INFO:mascaf.graph_fitting:Tracing start: mesh[V=34,F=64], skeleton[nodes=4,edges=3], spacing=1, radius_strategy=equivalent_area
INFO:mascaf.graph_fitting:Computing skeleton node radii using multi-tangent approach...
INFO:mascaf.graph_fitting:Computed radii for 4 skeleton nodes
INFO:mascaf.graph_fitting:Edge group 0: input_pts=4 -> samples=11
DEBUG:trimesh.util:contains_points executed in 0.0000 seconds.
DEBUG:mascaf.graph_fitting:pl=0 i=0 P=(1.503e-14,2.897e-15,4.908) r=0.9872 source=skeleton_node inside=True
DEBUG:trimesh.util:contains_points executed in 0.0000 seconds.
DEBUG:mascaf.graph_fitting:pl=0 i=1 P=(2.153e-05,-9.089e-05,3.908) r=0.9872 source=equivalent_area inside=True
DEBUG:trimesh.util:contains_points executed in 0.0000 seconds.
DEBUG:mascaf.graph_fitting:pl=0 i=2 P=(2.153e-05,-9.089e-05,2.908) r=0.9872 source=equivalent_area inside=True
DEBUG:trimesh.util:contains_points executed in 0.0000 seconds.
DEBUG:mascaf.graph_fitting:pl=0 i=3 P=(2.153e-05,-9.089e-05,1.908) r=0.987

MorphologyGraph: nodes=11, edges=10, components=1, cycles=0, branch_points=0, leaves=2, self_loops=0, density=0.1818


# Plot fitted SWC model

In [11]:
# plot using swctools
from swctools import SWCModel, plot_model

model = SWCModel.from_swc_file("../../data/demo/cylinder.swc")
plot_model(swc_model=model)

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=11 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_parse_result records=11 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_swc_file built nodes=11 edges=10 strict=True validate_reconnections=True
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geometry:frustum_mesh sides=16 end_caps=False verts=32 faces=32
DEBUG:swctools.geomet

In [12]:
from mascaf.validation import Validation

validator = Validation(mm, skel, morph)
vol_result = validator.compare_volumes()
area_result = validator.compare_surface_areas()

print("Validation Results:")
print("\nVolume Comparison:")
print(f"  Mesh volume:       {vol_result['mesh_volume']:.4f}")
print(f"  Morphology volume: {vol_result['morphology_volume']:.4f}")
print(f"  Ratio:             {vol_result['ratio']:.4f}")
print(f"  Absolute diff:     {vol_result['absolute_difference']:.4f}")
print(f"  Relative error:    {vol_result['relative_error']:.2%}")
print("\nSurface Area Comparison:")
print(f"  Mesh area:         {area_result['mesh_area']:.4f}")
print(f"  Morphology area:   {area_result['morphology_area']:.4f}")
print(f"  Ratio:             {area_result['ratio']:.4f}")
print(f"  Absolute diff:     {area_result['absolute_difference']:.4f}")
print(f"  Relative error:    {area_result['relative_error']:.2%}")

INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 34 vertices, 64 faces
INFO:mascaf.validation:  Skeleton: 4 nodes, 3 edges
INFO:mascaf.validation:  MorphologyGraph: 11 nodes, 10 edges


Validation Results:

Volume Comparison:
  Mesh volume:       30.6147
  Morphology volume: 30.0511
  Ratio:             0.9816
  Absolute diff:     0.5635
  Relative error:    1.84%

Surface Area Comparison:
  Mesh area:         68.5518
  Morphology area:   67.0066
  Ratio:             0.9775
  Absolute diff:     1.5452
  Relative error:    2.25%
